In [22]:
import torchvision 
import torch
import torchvision.transforms.v2 as T
from torch.utils.data import DataLoader
import torch.nn as nn
import torchmetrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [16]:
toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets", train=True, download=True, transform=toTensor)

test_data = torchvision.datasets.FashionMNIST(root="datasets", train=False, download=True, transform=toTensor)

torch.manual_seed(42)

train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

In [17]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

In [18]:
X_samle, y_sample = train_data[0]
print(X_samle.shape)
print(X_samle.dtype)
print(train_and_valid_data.classes[y_sample])

torch.Size([1, 28, 28])
torch.float32
Ankle boot


In [19]:
class ImageClassifier(nn.Module):
    def __init__(self, n_input, n_hidden1, n_hidden2, n_classes):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Flatten(),

            nn.Linear(n_input, n_hidden1),
            nn.ReLU(),
            nn.Linear(n_hidden1, n_hidden2),
            nn.ReLU(),
            nn.Linear(n_hidden2, n_classes)
        )
    def forward(self, X):
        return self.mlp(X)

torch.manual_seed(42)
model = ImageClassifier(28*28, 300, 100, 10)

xentropy = nn.CrossEntropyLoss()

In [20]:
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

In [21]:
model.to(device)
optimizer = torch.optim.Adam(model.parameters())

for epoch in range(10):
    model.train()
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        loss = xentropy(model(X), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        for X, y in valid_loader:
            X, y = X.to(device), y.to(device)
            accuracy.update(model(X), y)
    print(f'Epoch {epoch+1}: val_acc={accuracy.compute():.4f}')
    accuracy.reset()

Epoch 1: val_acc=0.8552
Epoch 2: val_acc=0.8662
Epoch 3: val_acc=0.8726
Epoch 4: val_acc=0.8690
Epoch 5: val_acc=0.8780
Epoch 6: val_acc=0.8802
Epoch 7: val_acc=0.8898
Epoch 8: val_acc=0.8846
Epoch 9: val_acc=0.8734
Epoch 10: val_acc=0.8858


In [26]:
model.eval()
X_new, y_new = next(iter(valid_loader))
X_new = X_new[:3].to(device)
with torch.no_grad():
    y_pred_logits = model(X_new)

y_pred = y_pred_logits.argmax(dim=1)
print(y_pred)
[train_and_valid_data.classes[index] for index in y_pred]

tensor([7, 4, 2], device='cuda:0')


['Sneaker', 'Coat', 'Pullover']

In [28]:
import torch.nn.functional as F

y_proba = F.softmax(y_pred_logits, dim=1)
y_proba.round(decimals=3)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0030, 0.0000, 0.9830, 0.0000,
         0.0130],
        [0.0000, 0.0000, 0.0360, 0.0000, 0.9630, 0.0000, 0.0010, 0.0000, 0.0000,
         0.0000],
        [0.0010, 0.0000, 0.5910, 0.0010, 0.3310, 0.0000, 0.0760, 0.0000, 0.0000,
         0.0000]], device='cuda:0')